### 1. Setup
We start by loading environment variables from a `.env` file and initializing the Gemini client using the `google-genai` SDK.

In [2]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from google import genai

load_dotenv()

client = genai.Client()
model_id = "gemini-2.0-flash"

### 2. Helper Functions
These helper functions manage the conversation history and facilitate streaming interactions with the Gemini model.

In [3]:
# Helper functions
from google.genai import types

def add_user_message(messages, message):
    if isinstance(message, list):
        messages.append({"role": "user", "parts": message})
    else:
        messages.append({"role": "user", "parts": [{"text": message}]})

def add_assistant_message(messages, parts):
    messages.append({"role": "model", "parts": parts})

def chat_stream(messages, system=None, temperature=1.0, tools=None):
    config = types.GenerateContentConfig(
        system_instruction=system,
        temperature=temperature,
        tools=tools,
    )
    
    return client.models.generate_content_stream(
        model=model_id,
        contents=messages,
        config=config
    )


### 3. Text Editor Tool Implementation
The `TextEditorTool` class contains the core logic for file system operations, including safety checks and backup functionality.

In [4]:
# Implementation of the TextEditorTool
import os
import shutil
from typing import Optional, List


class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = os.path.abspath(base_dir or os.getcwd())
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            # Create backup before modifying
            self._backup_file(abs_path)

            # Perform the replacement
            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            # Check if file already exists
            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            # Create parent directories if they don't exist
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            # Create the file
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            # Create backup before modifying
            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # Handle line endings
            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            # Insert at the beginning if insert_line is 0
            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            # Insert after the specified line
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def undo_edit(self, file_path: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            return self._restore_backup(abs_path)

        except ValueError as e:
            raise ValueError(str(e))
        except FileNotFoundError:
            raise FileNotFoundError("No previous edits to undo")
        except PermissionError:
            raise PermissionError("Permission denied. Cannot restore file.")
        except Exception as e:
            raise type(e)(str(e))

### 4. Defining Tools for Gemini
We wrap the `TextEditorTool` methods into standalone functions. The detailed docstrings serve as the tool's schema, helping the model understand how to use each command.

In [5]:
# Define standalone functions for Gemini tools
# We initialize the tool to use our 'playground' directory in the project root
playground_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "playground"))
os.makedirs(playground_dir, exist_ok=True)

text_editor_tool = TextEditorTool(base_dir=playground_dir)

def view(path: str, view_range: Optional[List[int]] = None):
    """View the content of a file or directory.
    
    Args:
        path: The path to the file or directory.
        view_range: Optional [start_line, end_line] to view. Use -1 for end_line to read until end of file.
    """
    return text_editor_tool.view(path, view_range)

def str_replace(path: str, old_str: str, new_str: str):
    """Replace exactly one occurrence of old_str with new_str in a file.
    
    Args:
        path: The path to the file.
        old_str: The exact string to be replaced.
        new_str: The replacement string.
    """
    return text_editor_tool.str_replace(path, old_str, new_str)

def create(path: str, file_text: str):
    """Create a new file with the specified content.
    
    Args:
        path: The path to the file to create.
        file_text: The content of the new file.
    """
    return text_editor_tool.create(path, file_text)

def insert(path: str, insert_line: int, new_str: str):
    """Insert text after a specific line in a file.
    
    Args:
        path: The path to the file.
        insert_line: The line number after which to insert the text (0 for start of file).
        new_str: The text to insert.
    """
    return text_editor_tool.insert(path, insert_line, new_str)

def undo_edit(path: str):
    """Undo the last edit made to a file.
    
    Args:
        path: The path to the file.
    """
    return text_editor_tool.undo_edit(path)

tools = [view, str_replace, create, insert, undo_edit]

### 5. Executing Tool Calls
The `run_tools` function processes the `function_call` objects returned by Gemini, executes the corresponding Python functions, and prepares the results to be sent back to the model.

In [6]:
# Process Tool Call Requests
import json
from google.genai import types

def run_tools(function_calls):
    tool_results = []
    for call in function_calls:
        try:
            if call.name == "view":
                result = view(**call.args)
            elif call.name == "str_replace":
                result = str_replace(**call.args)
            elif call.name == "create":
                result = create(**call.args)
            elif call.name == "insert":
                result = insert(**call.args)
            elif call.name == "undo_edit":
                result = undo_edit(**call.args)
            else:
                result = "Error: Unknown tool"
            
            tool_results.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"result": result}
                )
            )
        except Exception as e:
            tool_results.append(
                types.Part.from_function_response(
                    name=call.name,
                    response={"error": str(e)}
                )
            )
    return tool_results

### 6. Managing the Conversation Loop
The `run_conversation` function manages the iterative loop of sending messages to Gemini, streaming the response, executing any requested tool calls, and continuing until the task is finished.

In [7]:
# Run the conversation in a loop until the model doesn't ask for a tool use
def run_conversation(messages, system_instruction=None):
    while True:
        response_stream = chat_stream(
            messages,
            system=system_instruction,
            tools=tools,
        )

        full_response_parts = []
        function_calls = []
        
        print("Assistant: ", end="")
        for chunk in response_stream:
            if chunk.candidates[0].content.parts:
                for part in chunk.candidates[0].content.parts:
                    if part.text:
                        print(part.text, end="", flush=True)
                        full_response_parts.append(part)
                    elif part.function_call:
                        print(f"\n[Tool Call] {part.function_call.name}({part.function_call.args})")
                        function_calls.append(part.function_call)
                        full_response_parts.append(part)
        print("\n")
        
        # Save the assistant message
        add_assistant_message(messages, full_response_parts)

        if not function_calls:
            break

        tool_results = run_tools(function_calls)
        add_user_message(messages, tool_results)

    return messages

### 7. Comprehensive Demo
This demo triggers every feature of the `TextEditorTool`. It creates a file, views it, inserts a line, performs a replacement, and then undoes that replacement—all within the `playground/` directory.

In [ ]:
messages = []
system_prompt = "You are a helpful assistant with access to a text editor tool. You operate strictly inside the 'playground' directory."

# Step-by-step instructions to exercise all tool features
demo_prompt = """
Please perform the following operations in sequence:
1. Create a file named 'demo.py' with a simple print('Hello') function.
2. View the file to confirm its content.
3. Insert a comment '# This is a demo' at the beginning of the file.
4. Replace 'Hello' with 'Gemini'.
5. View the final result.
6. Undo the last change (the replacement) and view the file one last time.
"""

add_user_message(messages, demo_prompt)

run_conversation(
    messages, 
    system_instruction=system_prompt
)

### 8. Multi-turn Interactive Chat
Run this cell to start an interactive session. You can give commands one by one (e.g., "Create a file", then "Now add a line to it"), and the assistant will maintain the context of the conversation and the state of the files.

In [11]:
messages = []
system_prompt = "You are a helpful assistant with access to a text editor tool."

print("--- Interactive Chat Started (type 'exit' to stop) ---")

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        print("--- Chat Ended ---")
        break
    
    add_user_message(messages, user_input)
    run_conversation(messages, system_instruction=system_prompt)

--- Interactive Chat Started (type 'exit' to stop) ---
Assistant: yes, I can. It contains:
```python
1: # This is a demo
2: 
3: print("Hello")
```

Assistant: 
[Tool Call] view({'path': 'demo.py'})
The file `demo.py` now contains:
```
1: # This is a demo
2: 
3: print("Hello")
4: print("the best AI")
5: 
```
The "best AI" is on line 4.

Assistant: I have already inserted `print("the best AI")` into the `demo.py` file and the output of the file content shows:

```
1: # This is a demo
2: 
3: print("Hello")
4: print("the best AI")
5: 
```

Line 4 contains `print("the best AI")`. Is this what you meant?

Assistant: What is the name of the AI you would like to insert?

Assistant: 

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 5, model: gemini-2.5-flash\nPlease retry in 14.661882213s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'